# 📊 Answer Evaluation NLP Pipeline

This notebook demonstrates how we evaluate interview answers using NLP:
- **Semantic Similarity** using sentence-transformers
- **Keyword Extraction** using spaCy
- **Clarity Score** using textstat
- **Confidence Detection** using filler word analysis

In [ ]:
# Install dependencies (run once)
# !pip install sentence-transformers spacy textstat scikit-learn
# !python -m spacy download en_core_web_sm

import spacy
import textstat
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load('en_core_web_sm')
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Models loaded ✓')

In [ ]:
# Sample data
question = 'Tell me about a time you handled a difficult team conflict.'

ideal_answer = '''
In my final year project, two team members disagreed on the tech stack. I organized a meeting,
facilitated a pros/cons discussion, and we voted on the best approach. The project was delivered
on time and received an A grade. I learned that structured communication resolves most conflicts.
'''

weak_answer = 'Um, like, I just talked to them and it was fine I guess. So yeah.'

strong_answer = '''
During my internship, two teammates disagreed on database design. As the team lead, I scheduled
a 30-minute sync, asked each person to present their approach, then we evaluated trade-offs
together. We chose the hybrid approach and delivered the feature 2 days early. The conflict
actually led to a better solution than either original proposal.
'''

In [ ]:
# 1. Semantic Similarity Score
def semantic_score(answer, ideal):
    vecs = model.encode([answer, ideal])
    return round(float(cosine_similarity([vecs[0]], [vecs[1]])[0][0]), 3)

print('=== Semantic Similarity ===')
print(f'Weak answer:   {semantic_score(weak_answer, ideal_answer)}')
print(f'Strong answer: {semantic_score(strong_answer, ideal_answer)}')

In [ ]:
# 2. Keyword Analysis
BEHAVIORAL_KEYWORDS = ['team', 'challenge', 'result', 'action', 'situation',
                        'task', 'learned', 'improved', 'delivered', 'led', 'conflict']

def keyword_score(answer):
    answer_lower = answer.lower()
    found = [kw for kw in BEHAVIORAL_KEYWORDS if kw in answer_lower]
    missing = [kw for kw in BEHAVIORAL_KEYWORDS if kw not in answer_lower]
    score = len(found) / len(BEHAVIORAL_KEYWORDS)
    return round(score, 3), found, missing

print('=== Keyword Analysis ===')
for label, ans in [('Weak', weak_answer), ('Strong', strong_answer)]:
    score, found, missing = keyword_score(ans)
    print(f'{label}: score={score}, found={found[:4]}, missing={missing[:3]}')

In [ ]:
# 3. Clarity Score (Flesch Reading Ease)
def clarity_score(answer):
    score = textstat.flesch_reading_ease(answer)
    normalized = min(max(score / 80, 0), 1)
    return round(normalized, 3)

print('=== Clarity Score ===')
print(f'Weak:   {clarity_score(weak_answer)}')
print(f'Strong: {clarity_score(strong_answer)}')

In [ ]:
# 4. Confidence Score (Filler word detection)
FILLER_WORDS = ['um', 'uh', 'like', 'you know', 'basically', 'literally', 'so yeah', 'i mean']

def confidence_score(answer):
    words = answer.lower().split()
    filler_count = sum(1 for w in words if w in FILLER_WORDS)
    ratio = filler_count / max(len(words), 1)
    return round(1.0 - min(ratio * 5, 1.0), 3)

print('=== Confidence Score ===')
print(f'Weak:   {confidence_score(weak_answer)}')
print(f'Strong: {confidence_score(strong_answer)}')

In [ ]:
# 5. Combined Final Score (Weighted Average)
WEIGHTS = {'semantic': 0.35, 'keyword': 0.25, 'clarity': 0.25, 'confidence': 0.15}

def final_score(answer, ideal):
    scores = {
        'semantic': semantic_score(answer, ideal),
        'keyword': keyword_score(answer)[0],
        'clarity': clarity_score(answer),
        'confidence': confidence_score(answer),
    }
    total = sum(scores[k] * WEIGHTS[k] for k in scores)
    return round(total * 10, 1), scores

print('=== FINAL SCORES (out of 10) ===')
for label, ans in [('Weak', weak_answer), ('Strong', strong_answer)]:
    score, breakdown = final_score(ans, ideal_answer)
    print(f'{label} answer: {score}/10')
    for k, v in breakdown.items():
        print(f'  {k}: {v}')
    print()